In [1]:
import pandas as pd
import numpy as np

ckd_clean = pd.read_csv("ckd_clean.csv")

print("Shape:", ckd_clean.shape)
print("\nColumns:")
print(ckd_clean.columns.tolist())

Shape: (400, 25)

Columns:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']


In [2]:
print("Data Types:")
print(ckd_clean.dtypes)

print("\nMissing Values:")
print(ckd_clean.isnull().sum())

print("\nDuplicate Rows:")
print(ckd_clean.duplicated().sum())

print("\nTarget Distribution:")
print(ckd_clean["classification"].value_counts())

print("\nTarget Percentage:")
print(
    ckd_clean["classification"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Data Types:
age               float64
bp                float64
sg                float64
al                float64
su                float64
rbc                object
pc                 object
pcc                object
ba                 object
bgr               float64
bu                float64
sc                float64
sod               float64
pot               float64
hemo              float64
pcv               float64
wc                float64
rc                float64
htn                object
dm                 object
cad                object
appet              object
pe                 object
ane                object
classification     object
dtype: object

Missing Values:
age                 9
bp                 12
sg                 47
al                 46
su                 49
rbc               152
pc                 65
pcc                 4
ba                  4
bgr                44
bu                 19
sc                 17
sod                87
pot                88

In [3]:
X_ckd = ckd_clean.drop(columns=["classification"])
y_ckd = ckd_clean["classification"]

print("Features shape:", X_ckd.shape)
print("Target shape:", y_ckd.shape)

print("\nTarget distribution:")
print(y_ckd.value_counts())

Features shape: (400, 24)
Target shape: (400,)

Target distribution:
classification
ckd       250
notckd    150
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X_train_ckd, X_test_ckd, y_train_ckd, y_test_ckd = train_test_split(
    X_ckd,
    y_ckd,
    test_size=0.20,
    stratify=y_ckd,
    random_state=42
)

print("Training set:", X_train_ckd.shape)
print("Test set:", X_test_ckd.shape)

print("\nTraining target distribution:")
print(y_train_ckd.value_counts())

print("\nTest target distribution:")
print(y_test_ckd.value_counts())

Training set: (320, 24)
Test set: (80, 24)

Training target distribution:
classification
ckd       200
notckd    120
Name: count, dtype: int64

Test target distribution:
classification
ckd       50
notckd    30
Name: count, dtype: int64


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features_ckd = [
    "age", "bp", "sg", "al", "su",
    "bgr", "bu", "sc", "sod", "pot",
    "hemo", "pcv", "wc", "rc"
]

categorical_features_ckd = [
    "rbc", "pc", "pcc", "ba",
    "htn", "dm", "cad", "appet",
    "pe", "ane"
]

numeric_transformer_ckd = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_ckd = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        drop="if_binary"
    ))
])

preprocessor_ckd = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_ckd, numeric_features_ckd),
        ("cat", categorical_transformer_ckd, categorical_features_ckd)
    ]
)

print("Numeric features:", len(numeric_features_ckd))
print("Categorical features:", len(categorical_features_ckd))
print("Total input features:", len(numeric_features_ckd) + len(categorical_features_ckd))

Numeric features: 14
Categorical features: 10
Total input features: 24


In [6]:
from sklearn.linear_model import LogisticRegression

ckd_lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_ckd),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

ckd_lr_pipeline.fit(X_train_ckd, y_train_ckd)

print("CKD Logistic Regression trained successfully.")

CKD Logistic Regression trained successfully.


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Predictions
y_pred_ckd_lr = ckd_lr_pipeline.predict(X_test_ckd)

# Probability for CKD
ckd_class_index = list(ckd_lr_pipeline.classes_).index("ckd")
y_prob_ckd_lr = ckd_lr_pipeline.predict_proba(X_test_ckd)[:, ckd_class_index]

# Metrics
accuracy = accuracy_score(y_test_ckd, y_pred_ckd_lr)
precision = precision_score(y_test_ckd, y_pred_ckd_lr, pos_label="ckd")
recall = recall_score(y_test_ckd, y_pred_ckd_lr, pos_label="ckd")
f1 = f1_score(y_test_ckd, y_pred_ckd_lr, pos_label="ckd")
roc_auc = roc_auc_score(
    (y_test_ckd == "ckd").astype(int),
    y_prob_ckd_lr
)

print("CKD Logistic Regression Results")
print("--------------------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_ckd,
    y_pred_ckd_lr,
    labels=["notckd", "ckd"]
))

CKD Logistic Regression Results
--------------------------------
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000
ROC-AUC  : 1.0000

Confusion Matrix:
[[30  0]
 [ 0 50]]


In [8]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv_ckd = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring_ckd = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc"
}

cv_results_ckd_lr = cross_validate(
    ckd_lr_pipeline,
    X_ckd,
    y_ckd,
    cv=cv_ckd,
    scoring=scoring_ckd,
    n_jobs=-1
)

print("CKD Logistic Regression - 5-Fold Cross-Validation")
print("---------------------------------------------------")

for metric in scoring_ckd:
    scores = cv_results_ckd_lr[f"test_{metric}"]
    print(
        f"{metric.upper():<10}: "
        f"{scores.mean():.4f} ± {scores.std():.4f}"
    )

CKD Logistic Regression - 5-Fold Cross-Validation
---------------------------------------------------
ACCURACY  : 0.9925 ± 0.0100
PRECISION : 0.9918 ± 0.0121
RECALL    : 0.9927 ± 0.0090
F1        : 0.9920 ± 0.0106
ROC_AUC   : 1.0000 ± 0.0000


In [27]:
from sklearn.neighbors import KNeighborsClassifier

ckd_knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_ckd),
    ("model", KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
        n_jobs=-1
    ))
])

ckd_knn_pipeline.fit(X_train_ckd, y_train_ckd)

print("CKD KNN trained successfully.")

CKD KNN trained successfully.


In [28]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Predictions
y_pred_ckd_knn = ckd_knn_pipeline.predict(X_test_ckd)

# Probability for CKD
ckd_class_index_knn = list(ckd_knn_pipeline.classes_).index("ckd")
y_prob_ckd_knn = ckd_knn_pipeline.predict_proba(X_test_ckd)[:, ckd_class_index_knn]

# Metrics
accuracy = accuracy_score(y_test_ckd, y_pred_ckd_knn)
precision = precision_score(y_test_ckd, y_pred_ckd_knn, pos_label="ckd")
recall = recall_score(y_test_ckd, y_pred_ckd_knn, pos_label="ckd")
f1 = f1_score(y_test_ckd, y_pred_ckd_knn, pos_label="ckd")
roc_auc = roc_auc_score(
    (y_test_ckd == "ckd").astype(int),
    y_prob_ckd_knn
)

print("CKD KNN Results")
print("----------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_ckd,
    y_pred_ckd_knn,
    labels=["notckd", "ckd"]
))

CKD KNN Results
----------------
Accuracy : 0.9625
Precision: 1.0000
Recall   : 0.9400
F1 Score : 0.9691
ROC-AUC  : 0.9987

Confusion Matrix:
[[30  0]
 [ 3 47]]


In [29]:
cv_results_ckd_knn = cross_validate(
    ckd_knn_pipeline,
    X_ckd,
    y_ckd,
    cv=cv_ckd,
    scoring=scoring_ckd,
    n_jobs=-1
)

print("CKD KNN - 5-Fold Cross-Validation")
print("---------------------------------")

for metric in scoring_ckd:
    scores = cv_results_ckd_knn[f"test_{metric}"]
    print(
        f"{metric.upper():<10}: "
        f"{scores.mean():.4f} ± {scores.std():.4f}"
    )

CKD KNN - 5-Fold Cross-Validation
---------------------------------
ACCURACY  : 0.9675 ± 0.0127
PRECISION : 0.9606 ± 0.0146
RECALL    : 0.9740 ± 0.0102
F1        : 0.9660 ± 0.0132
ROC_AUC   : 0.9979 ± 0.0039


In [30]:
from sklearn.ensemble import RandomForestClassifier

ckd_rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_ckd),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

ckd_rf_pipeline.fit(X_train_ckd, y_train_ckd)

print("CKD Random Forest trained successfully.")

CKD Random Forest trained successfully.


In [31]:
# Predictions
y_pred_ckd_rf = ckd_rf_pipeline.predict(X_test_ckd)

# Probability for CKD
ckd_class_index_rf = list(ckd_rf_pipeline.classes_).index("ckd")
y_prob_ckd_rf = ckd_rf_pipeline.predict_proba(X_test_ckd)[:, ckd_class_index_rf]

# Metrics
accuracy = accuracy_score(y_test_ckd, y_pred_ckd_rf)
precision = precision_score(y_test_ckd, y_pred_ckd_rf, pos_label="ckd")
recall = recall_score(y_test_ckd, y_pred_ckd_rf, pos_label="ckd")
f1 = f1_score(y_test_ckd, y_pred_ckd_rf, pos_label="ckd")
roc_auc = roc_auc_score(
    (y_test_ckd == "ckd").astype(int),
    y_prob_ckd_rf
)

print("CKD Random Forest Results")
print("--------------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_ckd,
    y_pred_ckd_rf,
    labels=["notckd", "ckd"]
))

CKD Random Forest Results
--------------------------
Accuracy : 0.9750
Precision: 0.9615
Recall   : 1.0000
F1 Score : 0.9804
ROC-AUC  : 1.0000

Confusion Matrix:
[[28  2]
 [ 0 50]]


In [32]:
cv_results_ckd_rf = cross_validate(
    ckd_rf_pipeline,
    X_ckd,
    y_ckd,
    cv=cv_ckd,
    scoring=scoring_ckd,
    n_jobs=-1
)

print("CKD Random Forest - 5-Fold Cross-Validation")
print("--------------------------------------------")

for metric in scoring_ckd:
    scores = cv_results_ckd_rf[f"test_{metric}"]
    print(
        f"{metric.upper():<10}: "
        f"{scores.mean():.4f} ± {scores.std():.4f}"
    )

CKD Random Forest - 5-Fold Cross-Validation
--------------------------------------------
ACCURACY  : 0.9900 ± 0.0094
PRECISION : 0.9910 ± 0.0080
RECALL    : 0.9880 ± 0.0124
F1        : 0.9893 ± 0.0101
ROC_AUC   : 1.0000 ± 0.0000


In [9]:
import joblib

joblib.dump(
    ckd_lr_pipeline,
    "ckd_model.pkl"
)

print("CKD model saved successfully!")

CKD model saved successfully!


In [10]:
import joblib

model = joblib.load("ckd_model.pkl")

print(type(model))
print(model)

<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'bp', 'sg', 'al',
                                                   'su', 'bgr', 'bu', 'sc',
                                                   'sod', 'pot', 'hemo', 'pcv',
                                                   'wc', 'rc']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_f

In [34]:
import os

print("Exists:", os.path.exists("ckd_model.pkl"))
print("Size:", round(os.path.getsize("ckd_model.pkl") / 1024, 2), "KB")

Exists: True
Size: 8.35 KB


In [35]:
import joblib

loaded_ckd_model = joblib.load("ckd_model.pkl")

ckd_test_prediction = loaded_ckd_model.predict(
    X_test_ckd.iloc[:5]
)

print("Loaded CKD model:", type(loaded_ckd_model).__name__)
print("Sample predictions:", ckd_test_prediction)

Loaded CKD model: Pipeline
Sample predictions: ['ckd' 'ckd' 'notckd' 'notckd' 'notckd']


In [36]:
ckd_clean.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [37]:
from sklearn.ensemble import GradientBoostingClassifier

# Gradient Boosting pipeline
ckd_gb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_ckd),
    ("model", GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

print("CKD Gradient Boosting pipeline created successfully!")

CKD Gradient Boosting pipeline created successfully!


In [39]:
# Train Gradient Boosting
ckd_gb_pipeline.fit(
    X_train_ckd,
    y_train_ckd
)

# Predictions
y_pred_ckd_gb = ckd_gb_pipeline.predict(
    X_test_ckd
)

# Probability for CKD
ckd_class_index_gb = list(
    ckd_gb_pipeline.classes_
).index("ckd")

y_prob_ckd_gb = ckd_gb_pipeline.predict_proba(
    X_test_ckd
)[:, ckd_class_index_gb]

# Calculate metrics separately
gb_accuracy = accuracy_score(
    y_test_ckd,
    y_pred_ckd_gb
)

gb_precision = precision_score(
    y_test_ckd,
    y_pred_ckd_gb,
    average="macro"
)

gb_recall = recall_score(
    y_test_ckd,
    y_pred_ckd_gb,
    average="macro"
)

gb_f1 = f1_score(
    y_test_ckd,
    y_pred_ckd_gb,
    average="macro"
)

gb_roc_auc = roc_auc_score(
    (y_test_ckd == "ckd").astype(int),
    y_prob_ckd_gb
)

# Display results
print("CKD Gradient Boosting — Test Set")
print("---------------------------------")
print(f"Accuracy : {gb_accuracy:.4f}")
print(f"Precision: {gb_precision:.4f}")
print(f"Recall   : {gb_recall:.4f}")
print(f"F1 Score : {gb_f1:.4f}")
print(f"ROC-AUC  : {gb_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_ckd,
    y_pred_ckd_gb
))

CKD Gradient Boosting — Test Set
---------------------------------
Accuracy : 0.9750
Precision: 0.9808
Recall   : 0.9667
F1 Score : 0.9730
ROC-AUC  : 1.0000

Confusion Matrix:
[[50  0]
 [ 2 28]]


In [40]:
# Gradient Boosting — 5-Fold Cross-Validation

cv_results_ckd_gb = cross_validate(
    ckd_gb_pipeline,
    X_ckd,
    y_ckd,
    cv=cv_ckd,
    scoring=scoring_ckd,
    n_jobs=-1
)

print("CKD Gradient Boosting - 5-Fold Cross-Validation")
print("------------------------------------------------")

for metric in scoring_ckd:
    scores = cv_results_ckd_gb[f"test_{metric}"]

    print(
        f"{metric.replace('_', ' ').title():10s}: "
        f"{scores.mean():.4f} ± {scores.std():.4f}"
    )

CKD Gradient Boosting - 5-Fold Cross-Validation
------------------------------------------------
Accuracy  : 0.9900 ± 0.0094
Precision : 0.9910 ± 0.0080
Recall    : 0.9880 ± 0.0124
F1        : 0.9893 ± 0.0101
Roc Auc   : 0.9989 ± 0.0013


In [43]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# AdaBoost base estimator
ckd_base_tree = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

# AdaBoost pipeline
ckd_adaboost_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_ckd),
    ("model", AdaBoostClassifier(
        estimator=ckd_base_tree,
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    ))
])

print("CKD AdaBoost pipeline created successfully!")

CKD AdaBoost pipeline created successfully!


In [44]:
# Train AdaBoost
ckd_adaboost_pipeline.fit(
    X_train_ckd,
    y_train_ckd
)

# Predictions
y_pred_ckd_ada = ckd_adaboost_pipeline.predict(
    X_test_ckd
)

# Probability for CKD
ckd_class_index_ada = list(
    ckd_adaboost_pipeline.classes_
).index("ckd")

y_prob_ckd_ada = ckd_adaboost_pipeline.predict_proba(
    X_test_ckd
)[:, ckd_class_index_ada]


# Metrics
ada_accuracy = accuracy_score(
    y_test_ckd,
    y_pred_ckd_ada
)

ada_precision = precision_score(
    y_test_ckd,
    y_pred_ckd_ada,
    average="macro"
)

ada_recall = recall_score(
    y_test_ckd,
    y_pred_ckd_ada,
    average="macro"
)

ada_f1 = f1_score(
    y_test_ckd,
    y_pred_ckd_ada,
    average="macro"
)

ada_roc_auc = roc_auc_score(
    (y_test_ckd == "ckd").astype(int),
    y_prob_ckd_ada
)

print("CKD AdaBoost — Test Set")
print("-----------------------")
print(f"Accuracy : {ada_accuracy:.4f}")
print(f"Precision: {ada_precision:.4f}")
print(f"Recall   : {ada_recall:.4f}")
print(f"F1 Score : {ada_f1:.4f}")
print(f"ROC-AUC  : {ada_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_ckd,
    y_pred_ckd_ada
))

CKD AdaBoost — Test Set
-----------------------
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000
ROC-AUC  : 1.0000

Confusion Matrix:
[[50  0]
 [ 0 30]]


In [45]:
# AdaBoost — 5-Fold Cross-Validation

cv_results_ckd_ada = cross_validate(
    ckd_adaboost_pipeline,
    X_ckd,
    y_ckd,
    cv=cv_ckd,
    scoring=scoring_ckd,
    n_jobs=-1
)

print("CKD AdaBoost - 5-Fold Cross-Validation")
print("----------------------------------------")

for metric in scoring_ckd:
    scores = cv_results_ckd_ada[f"test_{metric}"]

    print(
        f"{metric.replace('_', ' ').title():10s}: "
        f"{scores.mean():.4f} ± {scores.std():.4f}"
    )

CKD AdaBoost - 5-Fold Cross-Validation
----------------------------------------
Accuracy  : 0.9900 ± 0.0094
Precision : 0.9873 ± 0.0117
Recall    : 0.9920 ± 0.0075
F1        : 0.9894 ± 0.0099
Roc Auc   : 0.9997 ± 0.0005


In [46]:
from xgboost import XGBClassifier

# XGBoost pipeline for CKD
ckd_xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_ckd),
    ("model", XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

print("CKD XGBoost pipeline created successfully!")

CKD XGBoost pipeline created successfully!


In [50]:
# ============================================================
# CKD XGBoost — Target Preparation
# ============================================================

# Convert CKD target labels to numeric values
# ckd    -> 1
# notckd -> 0

y_ckd_xgb = y_ckd.map({
    "notckd": 0,
    "ckd": 1
})

print("XGBoost target distribution:")
print(y_ckd_xgb.value_counts())

print("\nUnique target values:")
print(sorted(y_ckd_xgb.unique()))

XGBoost target distribution:
classification
1    250
0    150
Name: count, dtype: int64

Unique target values:
[0, 1]


In [51]:
from sklearn.model_selection import train_test_split

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X_ckd,
    y_ckd_xgb,
    test_size=0.20,
    random_state=42,
    stratify=y_ckd_xgb
)

print("Training samples:", X_train_xgb.shape[0])
print("Testing samples :", X_test_xgb.shape[0])

print("\nTraining target distribution:")
print(y_train_xgb.value_counts())

print("\nTesting target distribution:")
print(y_test_xgb.value_counts())

Training samples: 320
Testing samples : 80

Training target distribution:
classification
1    200
0    120
Name: count, dtype: int64

Testing target distribution:
classification
1    50
0    30
Name: count, dtype: int64


In [49]:
# Create numeric target only for XGBoost
# CKD = 1
# NOT CKD = 0

y_ckd_xgb = y_ckd.map({
    "notckd": 0,
    "ckd": 1
})

# Check conversion
print("Original labels:")
print(y_ckd.value_counts())

print("\nXGBoost labels:")
print(y_ckd_xgb.value_counts())

Original labels:
classification
ckd       250
notckd    150
Name: count, dtype: int64

XGBoost labels:
classification
1    250
0    150
Name: count, dtype: int64


In [52]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

ckd_xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_ckd),
    ("model", XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

print("Fresh CKD XGBoost pipeline created successfully!")

Fresh CKD XGBoost pipeline created successfully!


In [53]:
# Train XGBoost

ckd_xgb_pipeline.fit(
    X_train_xgb,
    y_train_xgb
)

print("CKD XGBoost trained successfully!")

CKD XGBoost trained successfully!


In [54]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Predictions
y_pred_xgb = ckd_xgb_pipeline.predict(X_test_xgb)

# Probability of CKD = class 1
y_prob_xgb = ckd_xgb_pipeline.predict_proba(
    X_test_xgb
)[:, 1]

# Metrics
xgb_accuracy = accuracy_score(
    y_test_xgb,
    y_pred_xgb
)

xgb_precision = precision_score(
    y_test_xgb,
    y_pred_xgb
)

xgb_recall = recall_score(
    y_test_xgb,
    y_pred_xgb
)

xgb_f1 = f1_score(
    y_test_xgb,
    y_pred_xgb
)

xgb_roc_auc = roc_auc_score(
    y_test_xgb,
    y_prob_xgb
)

print("CKD XGBoost — Test Set")
print("----------------------")
print(f"Accuracy : {xgb_accuracy:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall   : {xgb_recall:.4f}")
print(f"F1 Score : {xgb_f1:.4f}")
print(f"ROC-AUC  : {xgb_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_xgb,
    y_pred_xgb
))

CKD XGBoost — Test Set
----------------------
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000
ROC-AUC  : 1.0000

Confusion Matrix:
[[30  0]
 [ 0 50]]


In [55]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv_xgb = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring_xgb = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

xgb_cv_results = cross_validate(
    ckd_xgb_pipeline,
    X_ckd,
    y_ckd_xgb,
    cv=cv_xgb,
    scoring=scoring_xgb,
    n_jobs=-1
)

print("CKD XGBoost — 5-Fold Cross-Validation")
print("---------------------------------------")

print(
    f"Accuracy : "
    f"{xgb_cv_results['test_accuracy'].mean():.4f} ± "
    f"{xgb_cv_results['test_accuracy'].std():.4f}"
)

print(
    f"Precision: "
    f"{xgb_cv_results['test_precision'].mean():.4f} ± "
    f"{xgb_cv_results['test_precision'].std():.4f}"
)

print(
    f"Recall   : "
    f"{xgb_cv_results['test_recall'].mean():.4f} ± "
    f"{xgb_cv_results['test_recall'].std():.4f}"
)

print(
    f"F1       : "
    f"{xgb_cv_results['test_f1'].mean():.4f} ± "
    f"{xgb_cv_results['test_f1'].std():.4f}"
)

print(
    f"ROC-AUC  : "
    f"{xgb_cv_results['test_roc_auc'].mean():.4f} ± "
    f"{xgb_cv_results['test_roc_auc'].std():.4f}"
)

CKD XGBoost — 5-Fold Cross-Validation
---------------------------------------
Accuracy : 0.9875 ± 0.0112
Precision: 0.9884 ± 0.0154
Recall   : 0.9920 ± 0.0160
F1       : 0.9900 ± 0.0090
ROC-AUC  : 0.9985 ± 0.0017


In [56]:
from sklearn.ensemble import VotingClassifier

# Base models
ckd_vote_lr = LogisticRegression(
    max_iter=2000,
    random_state=42
)

ckd_vote_rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

ckd_vote_gb = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

ckd_vote_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# Voting ensemble
ckd_voting_model = VotingClassifier(
    estimators=[
        ("lr", ckd_vote_lr),
        ("rf", ckd_vote_rf),
        ("gb", ckd_vote_gb),
        ("xgb", ckd_vote_xgb)
    ],
    voting="soft"
)

# Complete pipeline
ckd_voting_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_ckd),
    ("model", ckd_voting_model)
])

print("CKD Voting Ensemble pipeline created successfully!")

CKD Voting Ensemble pipeline created successfully!


In [57]:
# Train Voting Ensemble

ckd_voting_pipeline.fit(
    X_train_ckd,
    y_train_ckd
)

# Predictions
y_pred_ckd_voting = ckd_voting_pipeline.predict(
    X_test_ckd
)

# Probability for CKD
voting_classes = list(
    ckd_voting_pipeline.classes_
)

ckd_class_index_voting = voting_classes.index("ckd")

y_prob_ckd_voting = ckd_voting_pipeline.predict_proba(
    X_test_ckd
)[:, ckd_class_index_voting]


# Metrics
voting_accuracy = accuracy_score(
    y_test_ckd,
    y_pred_ckd_voting
)

voting_precision = precision_score(
    y_test_ckd,
    y_pred_ckd_voting,
    average="macro"
)

voting_recall = recall_score(
    y_test_ckd,
    y_pred_ckd_voting,
    average="macro"
)

voting_f1 = f1_score(
    y_test_ckd,
    y_pred_ckd_voting,
    average="macro"
)

voting_roc_auc = roc_auc_score(
    (y_test_ckd == "ckd").astype(int),
    y_prob_ckd_voting
)


print("CKD Voting Ensemble — Test Set")
print("--------------------------------")
print(f"Accuracy : {voting_accuracy:.4f}")
print(f"Precision: {voting_precision:.4f}")
print(f"Recall   : {voting_recall:.4f}")
print(f"F1 Score : {voting_f1:.4f}")
print(f"ROC-AUC  : {voting_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_ckd,
    y_pred_ckd_voting
))

CKD Voting Ensemble — Test Set
--------------------------------
Accuracy : 0.9750
Precision: 0.9808
Recall   : 0.9667
F1 Score : 0.9730
ROC-AUC  : 1.0000

Confusion Matrix:
[[50  0]
 [ 2 28]]


In [58]:
# CKD Voting Ensemble — 5-Fold Cross-Validation

cv_results_ckd_voting = cross_validate(
    ckd_voting_pipeline,
    X_ckd,
    y_ckd,
    cv=cv_ckd,
    scoring=scoring_ckd,
    n_jobs=-1
)

print("CKD Voting Ensemble - 5-Fold Cross-Validation")
print("-----------------------------------------------")

for metric in scoring_ckd:
    scores = cv_results_ckd_voting[f"test_{metric}"]

    print(
        f"{metric.replace('_', ' ').title():10s}: "
        f"{scores.mean():.4f} ± {scores.std():.4f}"
    )

CKD Voting Ensemble - 5-Fold Cross-Validation
-----------------------------------------------
Accuracy  : 0.9875 ± 0.0112
Precision : 0.9879 ± 0.0120
Recall    : 0.9860 ± 0.0127
F1        : 0.9866 ± 0.0119
Roc Auc   : 0.9999 ± 0.0003
